# Benchmarking Ageas Classification on Zheng68K PBMCs

Reliable classification of terminal cell states is the foundation of the Ageas
fate-inference pipeline: the ensemble must first learn the fate-memory
signatures that separate mature cell types before it can transfer them to
progenitor cells. This example reproduces the classification benchmark from the
Ageas manuscript on the **Zheng68K** peripheral blood mononuclear cell (PBMC)
dataset — a demanding benchmark characterised by pronounced class imbalance and
high transcriptional similarity among closely related immune subtypes.

Following the manuscript, we use a **five-fold cross-validation** selection
strategy and report **accuracy** and **macro F1** on a held-out split. On the
full dataset, Ageas reaches a median accuracy of ~0.83 and a median macro F1 of
~0.72 across the five folds, matching or exceeding correlation-based (Seurat,
SingleR) and deep-learning (scANVI, TOSICA) baselines.

> **Note.** This notebook is configured for a CUDA GPU (`accelerator='cuda'`),
> matching the other Ageas examples. Training the full candidate panel over
> five folds on ~68k cells is compute-intensive; run it on a GPU machine to
> reproduce the benchmark end to end.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from ageas import Hangar, n_kfold_selection
from ageas.tool import Multimodal_Corpus

warnings.filterwarnings('ignore')

## 1. Load the Zheng68K PBMC dataset

The dataset ships with the Ageas repository under `data/`. FACS-sorted labels
in `adata.obs['celltype']` provide the ground-truth terminal cell identities.
Printing the class distribution highlights the strong imbalance — from >20k
cytotoxic T cells down to a handful of rare subtypes.

In [ ]:
# Path to the Zheng68K FACS-labelled dataset shipped in the Ageas repo (data/).
data_path = '../../data/LiuBIB_5_Zheng68K(PBMC)_FACS_label.h5ad'

adata = sc.read_h5ad(data_path)
adata.obs['celltype'] = adata.obs['celltype'].astype('category')
print(adata)
adata.obs['celltype'].value_counts()

## 2. Preprocess

We select the top 2,000 highly variable genes (Seurat v3, on raw counts) and
normalise total counts per cell — the same preprocessing used in the other
Ageas examples.

In [ ]:
sc.pp.highly_variable_genes(
    adata, n_top_genes=2000, subset=True, flavor='seurat_v3'
)
sc.pp.normalize_total(adata)
adata

## 3. Hold out a stratified test set

To score the benchmark we set aside 20% of cells as a held-out test set,
stratified by cell type so that every class — including the rare ones — is
represented in both splits.

In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(adata.n_obs),
    test_size=0.2,
    stratify=adata.obs['celltype'],
    random_state=42,
)
adata_train = adata[train_idx].copy()
adata_test = adata[test_idx].copy()
print('train:', adata_train.shape, '| test:', adata_test.shape)

## 4. Build the Ageas corpora and model panel

`Multimodal_Corpus` wraps each `AnnData` split for Ageas, using `celltype` as
the categorical label. The `Hangar` loads the candidate model panel (neural
networks and classical estimators) from a config folder.

In [ ]:
corpus_train = Multimodal_Corpus(
    adata=adata_train, label_key='celltype', backed=False
)
corpus_test = Multimodal_Corpus(
    adata=adata_test, label_key='celltype', backed=False
)

# Candidate model panel (logreg / mlp / resnet / rnn / svc) shipped with Ageas.
hangar = Hangar('../../data/configs/default_config_v1')
print('Candidate units:', len(hangar.units))

## 5. Five-fold cross-validation selection

`n_kfold_selection` runs a single **5-fold** cross-validation round over the
candidate panel, ranking units by test accuracy and retaining the strongest
ones into the final ensemble (`Deck`). Per-fold splits are class-stratified and
oversampling is enabled to counter the pronounced class imbalance.

In [ ]:
deck = n_kfold_selection(
    accelerator='cuda',
    query_dataset=corpus_train,
    hangar=hangar,
    kfold_selection_list=[5],        # single 5-fold cross-validation round
    monitor_metric='test.accuracy',
    stratified_kfold_test=True,      # preserve class proportions across folds
    stratified_kfold_valid=True,
    oversample_method='repeat',      # mitigate class imbalance
    retention_point=0.5,
    verbose=True,
)

## 6. Predict on the held-out set

The selected ensemble averages class probabilities across its surviving units.
`deck.predict` returns the probability matrix and the integer ground-truth
labels in the corpus order.

In [ ]:
all_preds, all_labels = deck.predict(query_dataset=corpus_test)

y_true = all_labels
y_pred = np.argmax(all_preds, axis=1)

## 7. Benchmark metrics

We report the two metrics used in the manuscript — overall **accuracy** and
**macro F1** (which weights every class equally and is therefore the more
informative score under class imbalance) — together with a per-class
breakdown.

In [ ]:
label_names = [
    corpus_train.label_dict[i] for i in range(len(corpus_train.label_dict))
]

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
print(f'Accuracy : {acc:.3f}')
print(f'Macro F1 : {f1_macro:.3f}')
print()
print(classification_report(
    y_true, y_pred,
    labels=range(len(label_names)),
    target_names=label_names,
))

## 8. Confusion matrix

The confusion matrix reveals where the ensemble confuses transcriptionally
similar immune subtypes (e.g. the closely related CD4+/CD8+ T-cell states).

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(label_names)))

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
    ax=ax, xticks_rotation='vertical', colorbar=False, cmap='Blues'
)
ax.set_title(f'Zheng68K benchmark  |  Acc={acc:.3f}, macro F1={f1_macro:.3f}')
plt.tight_layout()
plt.show()

## 9. Visualise predictions on a UMAP

Finally, we embed the held-out cells and colour them by their ground-truth and
Ageas-predicted labels for a qualitative view of the classifier's agreement.

In [ ]:
corpus_test.adata.obs['pred'] = pd.Categorical(
    [corpus_train.label_dict[i] for i in y_pred]
)

sc.pp.pca(corpus_test.adata)
sc.pp.neighbors(corpus_test.adata)
sc.tl.umap(corpus_test.adata)
sc.pl.umap(corpus_test.adata, color=['celltype', 'pred'], wspace=0.4)